In [ ]:
import empyrical as ep
import numpy as np
import pandas as pd
import pyfolio as pf
import pyfolio.timeseries as ts
from datetime import datetime,timedelta

In [ ]:
df = get_all_securities(types=['index'])
df[df.display_name.str.contains("")]

In [ ]:
df=get_all_securities(['fund'])
df[df.display_name.str.contains("红利低波")]

In [ ]:
code='512890.XSHG'
df = get_price(
    code, 
    end_date=datetime.now(),
    count=3*365,
    frequency='1d', 
    fields=['close'], 
    skip_paused=False, 
    fq='pre', 
    panel=False, 
    fill_paused=True)
returns = df["close"].pct_change().dropna()
print("年化收益:", ep.annual_return(returns))
print("年化波动:", ep.annual_volatility(returns))
print("Sharpe:", ep.sharpe_ratio(returns))
print("Sortino:", ep.sortino_ratio(returns))
print("最大回撤:", ep.max_drawdown(returns))
df.plot(figsize=(10,5),title=code)

In [ ]:
def fixed_get_max_drawdown_underwater(underwater):
    valley = underwater.index[np.argmin(underwater)]

    # Find first 0
    peak = underwater[:valley][underwater[:valley] == 0].index[-1]

    # Find last 0
    try:
        recovery = underwater[valley:][underwater[valley:] == 0].index[0]
    except IndexError:
        recovery = np.nan

    return peak, valley, recovery


ts.get_max_drawdown_underwater = fixed_get_max_drawdown_underwater

In [ ]:
pf.create_full_tear_sheet(returns)

### 组合

In [ ]:
codes = [
    ('513300.XSHG', '纳斯达克ETF',0.3),
    ('512890.XSHG', '红利低波',0.4),
    ('518880.XSHG', '黄金ETF',0.3),
]

### 组合成分股各自走势

In [ ]:
df_list=[]
for code,name,weight in codes:    
    df = get_price(
        code, 
        end_date=datetime.now(),
        count=3*365,
        frequency='1d', 
        fields=['close'], 
        skip_paused=False, 
        fq='pre', 
        panel=False, 
        fill_paused=True)
    df = df.rename(columns={'close': name})
    df_list.append(df)
all_df = pd.concat(df_list, axis=1)    

In [ ]:
all_df.plot(figsize=(10,5))

### 按权重每日再平衡组合收益

In [ ]:
# 假设：每日再平衡，该算法隐含了"每天收盘按权重调仓"的假设。
# 1. 算每个资产的日收益率
returns_dict = {}
for code,name,weight in codes:
    returns_dict[name] = all_df[name].pct_change()
# 2. 拼成一个 df（自动对齐日期，缺失会变 NaN）
returns = pd.concat(returns_dict, axis=1).dropna(how='any')
# 3. 权重（按比例归一化，3:4:3）
weights = pd.Series([x[2] for x in codes], index=returns.columns)
# 4. 组合日收益率 = 加权平均
port_ret = (returns * weights).sum(axis=1)
print("年化收益:", ep.annual_return(port_ret))
print("年化波动:", ep.annual_volatility(port_ret))
print("Sharpe:", ep.sharpe_ratio(port_ret))
print("Sortino:", ep.sortino_ratio(port_ret))
print("最大回撤:", ep.max_drawdown(port_ret))
port_ret_df = pd.DataFrame({'cumulative': (1 + port_ret).cumprod() - 1})
port_ret_df.plot(figsize=(10,5))

### 按权重不调仓组合收益

In [ ]:
# 如果是不再平衡（buy & hold，按初始资金配比后持有），那要换算法：
# 先用初始价格算出资金占比，再算每只资产的份额
init_prices = all_df.iloc[0]
shares = weights / init_prices        # 每只资产买多少"份"
holdings = df * shares                  # 每天持仓市值
nav = holdings.sum(axis=1)              # 总净值
bh_ret = nav.pct_change()               # buy & hold 日收益
print("年化收益:", ep.annual_return(bh_ret))
print("年化波动:", ep.annual_volatility(bh_ret))
print("Sharpe:", ep.sharpe_ratio(bh_ret))
print("Sortino:", ep.sortino_ratio(bh_ret))
print("最大回撤:", ep.max_drawdown(bh_ret))
bh_ret_df = pd.DataFrame({'cumulative': (1 + bh_ret).cumprod() - 1})
bh_ret_df.plot(figsize=(10,5))